# Stufe 2: Inter-Satz-Relationen

Dieses Notebook implementiert **Ebene 0, Stufe 2**: Die Bestimmung inferentieller
Relationen **zwischen Propositionen verschiedener Sätze**.

**Kapitelstruktur:** Inter-Satz-Relationen werden NUR innerhalb desselben Kapitels
bestimmt. Kapitelgrenzen werden nicht überschritten.

**Input:** Die JSON-Datei aus Stufe 1 (`*_stufe1.json`)
**Output:** Ein zusammengeführter Gesamt-DataFrame (Stufe 1 + 2), Export als CSV/XLSX

## 1. Setup

In [1]:
import pandas as pd
import os
import json
import re
import time
from pathlib import Path
from anthropic import Anthropic
from tqdm.notebook import tqdm
from collections import Counter, defaultdict

In [2]:
# ===== KONFIGURATION =====

ANTHROPIC_API_KEY = os.getenv('MY_ANTHROPIC')

# Input aus Stufe 1
STUFE1_JSON = "Beutin_complete_Direktzitat_stufe1.json"  # <-- Hier anpassen
STUFE1_CSV = "Beutin_complete_Direktzitat_stufe1.csv"    # <-- Hier anpassen

MODEL = "claude-opus-4-8"
API_DELAY = 1.0

STEM = Path(STUFE1_JSON).stem.replace('_stufe1', '')
OUTPUT_INTER_CSV = f"{STEM}_stufe2_inter.csv"
OUTPUT_GESAMT_CSV = f"{STEM}_gesamt.csv"
OUTPUT_GESAMT_XLSX = f"{STEM}_gesamt.xlsx"
OUTPUT_INTER_JSON = f"{STEM}_stufe2_inter.json"
# Checkpoint-Konfiguration
CHECKPOINT_DIR = f"{STEM}_checkpoints_stufe2"
CHECKPOINT_INTERVAL = 20  # Alle N Paare speichern


In [3]:
client = Anthropic(api_key=ANTHROPIC_API_KEY)

try:
    test = client.messages.create(
        model=MODEL,
        max_tokens=10,
        messages=[{"role": "user", "content": "Antworte nur mit: OK"}],
    )
    print(f"Verbindung OK. Modell: {MODEL}")
    print(f"Antwort: {test.content[0].text}")
except Exception as e:
    print(f"Fehler: {e}")

# Stufe-1-Daten laden
with open(STUFE1_JSON, 'r', encoding='utf-8') as f:
    stufe1_data = json.load(f)

print(f"\nGeladen: {len(stufe1_data)} S\u00e4tze aus Stufe 1")

# Nur S\u00e4tze mit Propositionen (keine Fragmente, keine _failed)
valid = [s for s in stufe1_data
         if s.get('propositionen') and not s.get('_failed', False)]
print(f"Davon mit Propositionen: {len(valid)}")

# Kapitelstruktur anzeigen
kapitel_counts = Counter(s.get('kapitel', '(Ohne)') for s in valid)
print(f"\nKapitel:")
for ch, count in kapitel_counts.items():
    print(f"  {ch[:50]}: {count} S\u00e4tze")


Verbindung OK. Modell: claude-opus-4-8
Antwort: OK

Geladen: 11610 Sätze aus Stufe 1
Davon mit Propositionen: 10870

Kapitel:
  Mittelalterliche Literatur: 815 Sätze
  (Unbekannt): 5 Sätze
  Humanismus und Reformation: 612 Sätze
  Literatur des Barock: 546 Sätze
  Aufklärung: 525 Sätze
  Kunstepoche: 820 Sätze
  Vormärz: 517 Sätze
  Realismus und Gründerzeit: 536 Sätze
  Die literarische Moderne (1890–1920): 689 Sätze
  Literatur in der Weimarer Republik: 620 Sätze
  Literatur im Dritten Reich: 232 Sätze
  Die deutsche Literatur des Exils: 326 Sätze
  Deutsche Literatur nach 1945: 406 Sätze
  Die Literatur der DDR: 1051 Sätze
  Die Literatur der Bundesrepublik: 1019 Sätze
  Tendenzen in der deutschsprachigen Gegenwartsliter: 2151 Sätze


## 2. Satzpaare bilden (kapitelweise)

Inter-Satz-Relationen werden nur zwischen aufeinanderfolgenden Sätzen
**innerhalb desselben Kapitels** gebildet. An Kapitelgrenzen wird kein Paar gebildet.

In [4]:
def build_chapter_pairs(valid_sentences):
    """Bildet Satzpaare nur innerhalb desselben Kapitels.

    ROBUST gegen Umsortierung durch Resume/Checkpoints: Die Saetze werden
    pro Kapitel nach ihrer satz_id-Nummer SORTIERT, bevor konsekutive Paare
    gebildet werden. Damit ist die Paarung unabhaengig von der Array-
    Reihenfolge im Stufe-1-Bestand. Es entstehen keine falschen Paare durch
    verschobene Eintraege, und an Resume-Grenzen fehlende Naht-Paare werden
    wieder eingesammelt. Die Kapitel selbst bleiben in Lesereihenfolge
    (erstes Auftreten).
    """
    def _num(sid):
        m = re.match(r'S-(\d+)', sid or '')
        return int(m.group(1)) if m else float('inf')  # nicht parsebare ans Ende

    # Kapitel in Reihenfolge des ersten Auftretens; Dubletten (z. B. S-11066)
    # werden anhand der satz_id entfernt (erstes Vorkommen behalten).
    chapter_order = []
    by_chapter = {}
    seen_ids = set()
    n_dup = 0
    for s in valid_sentences:
        sid = s.get('satz_id')
        if sid in seen_ids:
            n_dup += 1
            continue
        seen_ids.add(sid)
        ch = s.get('kapitel', '')
        if ch not in by_chapter:
            by_chapter[ch] = []
            chapter_order.append(ch)
        by_chapter[ch].append(s)

    pairs = []
    for ch in chapter_order:
        sents = sorted(by_chapter[ch], key=lambda s: _num(s.get('satz_id')))
        for i in range(len(sents) - 1):
            pairs.append((sents[i], sents[i + 1]))

    print(f"Satzpaare gesamt: {len(pairs)}")
    print(f"Kapitel: {len(chapter_order)}")
    if n_dup:
        print(f"Doppelte satz_id uebersprungen (erstes Vorkommen behalten): {n_dup}")

    ch_pair_counts = Counter(p[0].get('kapitel', '') for p in pairs)
    print(f"\nPaare pro Kapitel:")
    for ch in chapter_order:
        print(f"  {ch[:50]}: {ch_pair_counts.get(ch, 0)}")

    return pairs


pairs = build_chapter_pairs(valid)


Satzpaare gesamt: 10853
Kapitel: 16
Doppelte satz_id uebersprungen (erstes Vorkommen behalten): 1

Paare pro Kapitel:
  Mittelalterliche Literatur: 814
  (Unbekannt): 4
  Humanismus und Reformation: 611
  Literatur des Barock: 545
  Aufklärung: 524
  Kunstepoche: 819
  Vormärz: 516
  Realismus und Gründerzeit: 535
  Die literarische Moderne (1890–1920): 688
  Literatur in der Weimarer Republik: 619
  Literatur im Dritten Reich: 231
  Die deutsche Literatur des Exils: 325
  Deutsche Literatur nach 1945: 405
  Die Literatur der DDR: 1050
  Die Literatur der Bundesrepublik: 1018
  Tendenzen in der deutschsprachigen Gegenwartsliter: 2149


## 3. Inter-Satz-Relationen extrahieren

In [5]:
# System-Prompt: CoT-Fassung
PROMPT_FILE_INTER = 'prompt_inter-satz_cot.md'

if os.path.exists(PROMPT_FILE_INTER):
    raw = open(PROMPT_FILE_INTER, 'r', encoding='utf-8').read()
    if raw.strip().startswith('SYSTEM_PROMPT'):
        start = raw.index('"""') + 3
        end = raw.rindex('"""')
        SYSTEM_PROMPT_INTER = raw[start:end]
    else:
        SYSTEM_PROMPT_INTER = raw
    print(f"Inter-Satz-Prompt geladen: {len(SYSTEM_PROMPT_INTER)} Zeichen")
else:
    print(f"WARNUNG: {PROMPT_FILE_INTER} nicht gefunden!")
    SYSTEM_PROMPT_INTER = "PLACEHOLDER"

print(f"Prompt-Anfang: {SYSTEM_PROMPT_INTER[:100]}...")

Inter-Satz-Prompt geladen: 11947 Zeichen
Prompt-Anfang: Du bist ein Annotationswerkzeug für die inferentielle Analyse von Literaturgeschichten nach Robert B...


In [6]:
def parse_inter_response(raw, sid_a, sid_b):
    """Tolerantes Parsen der API-Antwort.

    Behebt 'Extra data'-Fehler: Wenn das Modell nach dem JSON-Objekt noch
    weiteren Text oder ein zweites Objekt anhaengt, sammelt diese Funktion
    ALLE Top-Level-JSON-Objekte ein und fuehrt ihre inter_relationen
    zusammen (dedupliziert nach von/nach/typ). Unvollstaendiges JSON
    (z. B. durch max_tokens-Abschnitt) fuehrt weiterhin zu einem Fehler
    und damit zu einem Retry.
    """
    dec = json.JSONDecoder()
    s = (raw or "").strip()
    idx, objs = 0, []
    while idx < len(s):
        while idx < len(s) and s[idx] in " \t\r\n":
            idx += 1
        if idx >= len(s):
            break
        try:
            obj, end = dec.raw_decode(s, idx)
        except json.JSONDecodeError:
            break  # nicht-JSON-Reststueck am Ende ignorieren
        objs.append(obj)
        idx = end
    # In Listen verpackte Objekte flachklopfen
    flat = []
    for o in objs:
        flat.extend(o if isinstance(o, list) else [o])
    dicts = [o for o in flat if isinstance(o, dict)]
    if not dicts:
        raise json.JSONDecodeError("kein JSON-Objekt gefunden", s, 0)
    base = dicts[0]
    seen, merged = set(), []
    for o in dicts:
        for r in o.get("inter_relationen", []):
            key = (r.get("von"), r.get("nach"), r.get("typ"))
            if key not in seen:
                seen.add(key)
                merged.append(r)
    return {
        "satz_a": base.get("satz_a", sid_a),
        "satz_b": base.get("satz_b", sid_b),
        "inter_relationen": merged,
    }


def build_pair_prompt(satz_a, satz_b):
    def format_satz(s):
        sid = s['satz_id']
        props = s.get('propositionen', [])
        lines = [f"{p['id']}: {p['text']}" for p in props]
        return f"{sid}\nPropositionen:\n" + "\n".join(lines)
    return (
        f"Bestimme die Inter-Satz-Relationen zwischen Satz A und Satz B.\n\n"
        f"=== Satz A ===\n{format_satz(satz_a)}\n\n"
        f"=== Satz B ===\n{format_satz(satz_b)}\n\n"
        f"WICHTIG: Antworte NUR mit dem JSON-Objekt. "
        f"Beginne direkt mit {{ und ende mit }}. "
        f"Kein Text davor, kein Text danach, kein Markdown, keine Erkl\u00e4rungen."
    )


def call_inter_api(satz_a, satz_b, max_retries=3):
    prompt = build_pair_prompt(satz_a, satz_b)
    sid_a = satz_a['satz_id']
    sid_b = satz_b['satz_id']
    for attempt in range(max_retries):
        try:
            resp = client.messages.create(
                model=MODEL,
                max_tokens=4096,  # Headroom fuer ausfuehrliches CoT-Reasoning (vorher 2048)
                system=SYSTEM_PROMPT_INTER,
                messages=[{"role": "user", "content": prompt}],
            )
            raw = resp.content[0].text.strip() if resp.content else ""

            if not raw:
                tqdm.write(f"  WARNUNG: Leere API-Antwort ({sid_a}/{sid_b}, Versuch {attempt+1}/{max_retries})")
                if attempt < max_retries - 1: time.sleep(5)
                continue

            if resp.stop_reason == 'max_tokens':
                tqdm.write(f"  WARNUNG: Antwort abgeschnitten ({sid_a}/{sid_b}, Versuch {attempt+1})")

            if raw.startswith("```"):
                raw = re.sub(r'^```(?:json)?\s*', '', raw)
                raw = re.sub(r'\s*```$', '', raw)
            # Toleranter Parser statt json.loads: faengt 'Extra data' ab und
            # fuehrt mehrere Objekte zusammen, ohne Relationen zu verlieren.
            return parse_inter_response(raw, sid_a, sid_b)
        except json.JSONDecodeError as e:
            tqdm.write(f"  JSON-Fehler ({sid_a}/{sid_b}, Versuch {attempt+1}/{max_retries}): {e}")
            tqdm.write(f"    Raw-Antwort ({len(raw)} Zeichen): {raw[:200]}")
            if attempt < max_retries - 1: time.sleep(2)
        except Exception as e:
            error_type = type(e).__name__
            tqdm.write(f"  {error_type} ({sid_a}/{sid_b}, Versuch {attempt+1}/{max_retries}): {e}")
            if 'rate' in str(e).lower() or 'overloaded' in str(e).lower():
                wait = 30 if attempt == 0 else 60
                tqdm.write(f"    Rate-Limit/Overloaded \u2014 warte {wait}s")
                time.sleep(wait)
            elif attempt < max_retries - 1:
                time.sleep(5)
    tqdm.write(f"  FEHLER: Paar {sid_a}/{sid_b} nach {max_retries} Versuchen gescheitert")
    return {"satz_a": sid_a, "satz_b": sid_b, "inter_relationen": [], "_failed": True}


def save_checkpoint_inter(results, pair_idx, checkpoint_dir):
    os.makedirs(checkpoint_dir, exist_ok=True)
    cp_path = os.path.join(checkpoint_dir, f"checkpoint_pair_{pair_idx:04d}.json")
    with open(cp_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    latest_path = os.path.join(checkpoint_dir, "checkpoint_latest.json")
    with open(latest_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    return cp_path


def load_checkpoint_inter(checkpoint_dir):
    latest_path = os.path.join(checkpoint_dir, "checkpoint_latest.json")
    if os.path.exists(latest_path):
        with open(latest_path, 'r', encoding='utf-8') as f:
            results = json.load(f)
        processed_pairs = {(r.get('satz_a',''), r.get('satz_b',''))
                           for r in results if not r.get('_failed', False)}
        n_failed = sum(1 for r in results if r.get('_failed', False))
        results = [r for r in results if not r.get('_failed', False)]
        print(f"Checkpoint geladen: {len(results)} Paare erfolgreich verarbeitet")
        if n_failed > 0:
            print(f"  {n_failed} fehlgeschlagene Paare werden erneut versucht")
        return results, processed_pairs
    return [], set()


In [7]:
# Verarbeitung mit Checkpoint-Support
inter_results, processed_pairs = load_checkpoint_inter(CHECKPOINT_DIR)

remaining_pairs = [(a, b) for a, b in pairs
                   if (a['satz_id'], b['satz_id']) not in processed_pairs]

if not remaining_pairs:
    print("Alle Paare bereits verarbeitet.")
else:
    print(f"Zu verarbeiten: {len(remaining_pairs)} von {len(pairs)} Paaren")
    if processed_pairs:
        print(f"  ({len(processed_pairs)} aus Checkpoint \u00fcbernommen)")
    
    for pair_idx, (satz_a, satz_b) in enumerate(tqdm(remaining_pairs,
                                                      desc="Stufe 2: Inter-Satz-Relationen")):
        result = call_inter_api(satz_a, satz_b)
        result['kapitel'] = satz_a.get('kapitel', '')
        inter_results.append(result)
        
        if (pair_idx + 1) % CHECKPOINT_INTERVAL == 0:
            cp = save_checkpoint_inter(inter_results, pair_idx, CHECKPOINT_DIR)
            tqdm.write(f"  Checkpoint: {len(inter_results)} Paare \u2192 {cp}")
        
        time.sleep(API_DELAY)
    
    save_checkpoint_inter(inter_results, len(remaining_pairs), CHECKPOINT_DIR)

# Statistik
n_failed = sum(1 for r in inter_results if r.get('_failed', False))
n_success = len(inter_results) - n_failed
total_inter = sum(len(r.get('inter_relationen', [])) for r in inter_results
                  if not r.get('_failed', False))
pairs_with_rel = sum(1 for r in inter_results
                     if r.get('inter_relationen') and not r.get('_failed', False))

print(f"\nVerarbeitet: {n_success} Satzpaare")
print(f"Paare mit Relationen: {pairs_with_rel} ({pairs_with_rel/max(n_success,1)*100:.1f}%)")
print(f"Inter-Satz-Relationen gesamt: {total_inter}")
if n_failed > 0:
    print(f"FEHLGESCHLAGEN: {n_failed} (werden bei erneutem Lauf erneut versucht)")


Checkpoint geladen: 4880 Paare erfolgreich verarbeitet
Zu verarbeiten: 5973 von 10853 Paaren
  (4880 aus Checkpoint übernommen)


Stufe 2: Inter-Satz-Relationen:   0%|          | 0/5973 [00:00<?, ?it/s]

  Checkpoint: 4900 Paare → Beutin_complete_Direktzitat_checkpoints_stufe2/checkpoint_pair_0019.json
  Checkpoint: 4920 Paare → Beutin_complete_Direktzitat_checkpoints_stufe2/checkpoint_pair_0039.json
  Checkpoint: 4940 Paare → Beutin_complete_Direktzitat_checkpoints_stufe2/checkpoint_pair_0059.json
  Checkpoint: 4960 Paare → Beutin_complete_Direktzitat_checkpoints_stufe2/checkpoint_pair_0079.json
  Checkpoint: 4980 Paare → Beutin_complete_Direktzitat_checkpoints_stufe2/checkpoint_pair_0099.json
  Checkpoint: 5000 Paare → Beutin_complete_Direktzitat_checkpoints_stufe2/checkpoint_pair_0119.json
  Checkpoint: 5020 Paare → Beutin_complete_Direktzitat_checkpoints_stufe2/checkpoint_pair_0139.json
  Checkpoint: 5040 Paare → Beutin_complete_Direktzitat_checkpoints_stufe2/checkpoint_pair_0159.json
  Checkpoint: 5060 Paare → Beutin_complete_Direktzitat_checkpoints_stufe2/checkpoint_pair_0179.json
  Checkpoint: 5080 Paare → Beutin_complete_Direktzitat_checkpoints_stufe2/checkpoint_pair_0199.json


In [8]:
# === Postprocessing 1: Ungültige Relationen entfernen (typ=null) ===
# Entfernt Einträge mit typ=null, typ="" oder typ=None aus inter_relationen.
n_removed = 0
for item in inter_results:
    rels = item.get('inter_relationen', [])
    clean_rels = [r for r in rels if r.get('typ') and r['typ'] != 'null']
    n_removed += len(rels) - len(clean_rels)
    item['inter_relationen'] = clean_rels
print(f"Postprocessing 1 (typ=null): {n_removed} entfernt" if n_removed
      else "Postprocessing 1 (typ=null): nichts zu entfernen")


# === Postprocessing 2: Intra-Satz-Relationen entfernen ===
# Eine Inter-Satz-Relation muss eine Proposition aus Satz A mit einer aus
# Satz B verbinden. Relationen, deren "von" und "nach" aus DEMSELBEN Satz
# stammen, gehören zu Stufe 1 und werden hier entfernt.
def _satznr(pid):
    m = re.match(r'P\((\d+)\.', pid or '')
    return m.group(1) if m else None

n_intra = 0
for item in inter_results:
    rels = item.get('inter_relationen', [])
    clean_rels = [r for r in rels
                  if not (_satznr(r.get('von'))
                          and _satznr(r.get('von')) == _satznr(r.get('nach')))]
    n_intra += len(rels) - len(clean_rels)
    item['inter_relationen'] = clean_rels
print(f"Postprocessing 2 (Intra-Satz): {n_intra} entfernt" if n_intra
      else "Postprocessing 2 (Intra-Satz): nichts zu entfernen")


# === Postprocessing 3: Reasoning-Konsistenz prüfen ===
# Verwirft Relationen, deren Reasoning sich selbst widerspricht: Wenn die
# LETZTE "→ ..."-Schlussfolgerung im Reasoning negativ ist ("Kein PR-...")
# oder nicht mit dem zugewiesenen typ übereinstimmt, hat das Modell laut
# eigenem Reasoning auf "kein Paar" geschlossen, die Relation aber trotzdem
# ausgegeben. Solche Fälle werden entfernt.
# Konservativ: Lässt sich keine Schlussfolgerung parsen, wird die Relation
# beibehalten (kein Fehlalarm).
def _letzte_schlussfolgerung(reasoning):
    cs = re.findall(r'→\s*((?:[Kk]ein\s+)?PR-[A-Z]+)', reasoning or '')
    return cs[-1] if cs else None

n_inkonsistent = 0
beispiele = []
for item in inter_results:
    rels = item.get('inter_relationen', [])
    clean_rels = []
    for r in rels:
        typ = r.get('typ')
        lc = _letzte_schlussfolgerung(r.get('reasoning'))
        if lc is None:
            clean_rels.append(r)  # nicht parsebar -> beibehalten
        elif typ and not lc.lower().startswith('kein') and lc == typ:
            clean_rels.append(r)  # konsistent
        else:
            n_inkonsistent += 1
            if len(beispiele) < 5:
                beispiele.append(f"{item.get('satz_a')}/{item.get('satz_b')} "
                                 f"[{typ}] letzte Schlussfolgerung: {lc!r}")
    item['inter_relationen'] = clean_rels

if n_inkonsistent:
    print(f"Postprocessing 3 (Reasoning-Konsistenz): {n_inkonsistent} entfernt "
          f"(Reasoning schloss auf 'kein Paar')")
    for b in beispiele:
        print(f"    - {b}")
else:
    print("Postprocessing 3 (Reasoning-Konsistenz): nichts zu entfernen")


# === Übersicht ===
gesamt_rel = sum(len(item.get('inter_relationen', [])) for item in inter_results)
leere_paare = sum(1 for item in inter_results if not item.get('inter_relationen'))
print(f"\nNach Postprocessing: {gesamt_rel} Relationen, "
      f"{leere_paare}/{len(inter_results)} leere Paare")


Postprocessing 1 (typ=null): 30 entfernt
Postprocessing 2 (Intra-Satz): 80 entfernt
Postprocessing 3 (Reasoning-Konsistenz): 660 entfernt (Reasoning schloss auf 'kein Paar')
    - S-002/S-003 [PR-BER] letzte Schlussfolgerung: 'kein PR-INK'
    - S-038/S-039 [PR-BER] letzte Schlussfolgerung: 'kein PR-INK'
    - S-052/S-053 [PR-INK] letzte Schlussfolgerung: 'kein PR-INK'
    - S-058/S-059 [PR-BER] letzte Schlussfolgerung: 'kein PR-INK'
    - S-068/S-069 [PR-BER] letzte Schlussfolgerung: 'kein PR-INK'

Nach Postprocessing: 5814 Relationen, 5330/10853 leere Paare


## 4. DataFrames erstellen

In [9]:
def inter_to_dataframe(inter_results):
    rows = []
    for item in inter_results:
        satz_a = item.get('satz_a', '')
        satz_b = item.get('satz_b', '')
        kapitel = item.get('kapitel', '')
        rels = item.get('inter_relationen', [])
        if not rels:
            rows.append({'Kapitel': kapitel, 'Satz A': satz_a, 'Satz B': satz_b,
                         'Von': '', 'Nach': '', 'Relationstyp': '[Keine Relation]',
                         'Konnektor': '', 'Reasoning': ''})
        else:
            for rel in rels:
                rows.append({'Kapitel': kapitel, 'Satz A': satz_a, 'Satz B': satz_b,
                             'Von': rel.get('von', ''), 'Nach': rel.get('nach', ''),
                             'Relationstyp': rel.get('typ', ''),
                             'Konnektor': rel.get('konnektor', ''),
                             'Reasoning': rel.get('reasoning', '')})
    return pd.DataFrame(rows)

df_inter = inter_to_dataframe(inter_results)
print(f"Inter-Satz-Relationen DataFrame: {len(df_inter)} Zeilen")
df_inter.head(10)

Inter-Satz-Relationen DataFrame: 11144 Zeilen


,Kapitel,Satz A,Satz B,Von,Nach,Relationstyp,Konnektor,Reasoning
0,Mittelalterliche Literatur,S-001,S-002,P(002.2),P(001.2),PR-BER,implizit (Satzübergang),Festlegungstest: Kann man P(002.2) (die Weltge...
1,Mittelalterliche Literatur,S-002,S-003,,,[Keine Relation],,
2,Mittelalterliche Literatur,S-003,S-004,,,[Keine Relation],,
3,Mittelalterliche Literatur,S-004,S-005,P(004.3),P(005.1),PR-BER,implizit (Satzübergang),Festlegungstest: Kann man P(004.3) (die muslim...
4,Mittelalterliche Literatur,S-004,S-005,P(004.3),P(005.2),PR-BER,implizit (Satzübergang),Festlegungstest: Kann man P(004.3) (die Araber...
5,Mittelalterliche Literatur,S-005,S-006,,,[Keine Relation],,
6,Mittelalterliche Literatur,S-006,S-007,,,[Keine Relation],,
7,Mittelalterliche Literatur,S-007,S-008,,,[Keine Relation],,
8,Mittelalterliche Literatur,S-008,S-009,,,[Keine Relation],,
9,Mittelalterliche Literatur,S-009,S-010,,,[Keine Relation],,


## 5. Zusammenführung mit Stufe 1

In [10]:
df_stufe1 = pd.read_csv(STUFE1_CSV, encoding='utf-8-sig')
print(f"Stufe 1: {len(df_stufe1)} Zeilen")


def merge_inter_relations(df_stufe1, inter_results):
    inter_by_prop = {}
    inter_typen_by_prop = {}
    for item in inter_results:
        for rel in item.get('inter_relationen', []):
            von = rel.get('von', '')
            nach = rel.get('nach', '')
            typ = rel.get('typ', '')
            konnektor = rel.get('konnektor', '')
            rel_str = f"{von} \u2192 {nach}: {typ} | {konnektor}"
            for pid in [von, nach]:
                if pid not in inter_by_prop:
                    inter_by_prop[pid] = []
                    inter_typen_by_prop[pid] = set()
                inter_by_prop[pid].append(rel_str)
                inter_typen_by_prop[pid].add(typ)
    df = df_stufe1.copy()
    df['Inter-Satz-Relationen'] = df['Propositions-ID'].map(
        lambda pid: '; '.join(inter_by_prop.get(pid, [])) if pid else '')
    df['Inter-Satz-Typen'] = df['Propositions-ID'].map(
        lambda pid: ', '.join(sorted(inter_typen_by_prop.get(pid, set()))) if pid else '')
    return df


df_gesamt = merge_inter_relations(df_stufe1, inter_results)
print(f"Gesamt: {len(df_gesamt)} Zeilen, {len(df_gesamt.columns)} Spalten")

Stufe 1: 29448 Zeilen
Gesamt: 29448 Zeilen, 10 Spalten


## 6. Export

In [11]:
df_inter.to_csv(OUTPUT_INTER_CSV, index=False, encoding='utf-8-sig')
print(f"Inter CSV: {OUTPUT_INTER_CSV}")

with open(OUTPUT_INTER_JSON, 'w', encoding='utf-8') as f:
    json.dump(inter_results, f, ensure_ascii=False, indent=2)
print(f"Inter JSON: {OUTPUT_INTER_JSON}")

df_gesamt.to_csv(OUTPUT_GESAMT_CSV, index=False, encoding='utf-8-sig')
print(f"Gesamt CSV: {OUTPUT_GESAMT_CSV}")

with pd.ExcelWriter(OUTPUT_GESAMT_XLSX, engine='openpyxl') as writer:
    df_gesamt.to_excel(writer, index=False, sheet_name='Propositionen')
    df_inter.to_excel(writer, index=False, sheet_name='Inter-Satz-Relationen')
    from openpyxl.styles import Font, PatternFill, Alignment
    hf = PatternFill(start_color='2B4C7E', end_color='2B4C7E', fill_type='solid')
    for ws in writer.sheets.values():
        for cell in ws[1]:
            cell.fill = hf
            cell.font = Font(name='Arial', size=10, bold=True, color='FFFFFF')
            cell.alignment = Alignment(wrap_text=True, vertical='top')
        ws.auto_filter.ref = ws.dimensions
print(f"Gesamt XLSX: {OUTPUT_GESAMT_XLSX}")

Inter CSV: Beutin_complete_Direktzitat_stufe2_inter.csv
Inter JSON: Beutin_complete_Direktzitat_stufe2_inter.json
Gesamt CSV: Beutin_complete_Direktzitat_gesamt.csv
Gesamt XLSX: Beutin_complete_Direktzitat_gesamt.xlsx


## 7. Statistiken

In [12]:
intra_total = sum(len(r.get('intra_relationen', [])) for r in stufe1_data)
inter_total = sum(len(r.get('inter_relationen', [])) for r in inter_results)

print(f"Gesamtstatistik Ebene 0")
print(f"{'='*50}")
print(f"Sätze:                     {len(stufe1_data)}")
print(f"Propositionen:             {sum(len(r.get('propositionen',[])) for r in stufe1_data)}")
print(f"Intra-Satz-Relationen:     {intra_total}")
print(f"Inter-Satz-Relationen:     {inter_total}")
print(f"Relationen gesamt:         {intra_total + inter_total}")
print()

# Typ-Verteilung
all_rels = []
for r in stufe1_data:
    for rel in r.get('intra_relationen', []):
        all_rels.append(('intra', rel.get('typ','')))
for r in inter_results:
    for rel in r.get('inter_relationen', []):
        all_rels.append(('inter', rel.get('typ','')))

if all_rels:
    typ_counter = Counter(t for _, t in all_rels)
    print(f"Relationstypen (gesamt):")
    for typ, count in typ_counter.most_common():
        intra_c = sum(1 for s, t in all_rels if s == 'intra' and t == typ)
        inter_c = sum(1 for s, t in all_rels if s == 'inter' and t == typ)
        print(f"  {typ}: {count:4d} (intra: {intra_c}, inter: {inter_c})")

# Pro Kapitel
print(f"\nPro Kapitel:")
for ch in dict.fromkeys(r.get('kapitel','') for r in inter_results):
    ch_pairs = [r for r in inter_results if r.get('kapitel') == ch]
    ch_rels = sum(len(r.get('inter_relationen', [])) for r in ch_pairs)
    ch_with = sum(1 for r in ch_pairs if r.get('inter_relationen'))
    print(f"  {ch[:50]}: {len(ch_pairs)} Paare, {ch_rels} Rels, {ch_with} mit Rel ({ch_with/max(len(ch_pairs),1)*100:.0f}%)")

Gesamtstatistik Ebene 0
Sätze:                     11610
Propositionen:             29448
Intra-Satz-Relationen:     9810
Inter-Satz-Relationen:     5814
Relationen gesamt:         15624

Relationstypen (gesamt):
  PR-BER: 14183 (intra: 8503, inter: 5680)
  PR-INK:  907 (intra: 863, inter: 44)
  PR-FLG:  523 (intra: 433, inter: 90)
  PR-UNA:   11 (intra: 11, inter: 0)

Pro Kapitel:
  Mittelalterliche Literatur: 814 Paare, 377 Rels, 358 mit Rel (44%)
  (Unbekannt): 4 Paare, 0 Rels, 0 mit Rel (0%)
  Humanismus und Reformation: 611 Paare, 332 Rels, 312 mit Rel (51%)
  Literatur des Barock: 545 Paare, 315 Rels, 294 mit Rel (54%)
  Aufklärung: 524 Paare, 382 Rels, 353 mit Rel (67%)
  Kunstepoche: 819 Paare, 480 Rels, 456 mit Rel (56%)
  Vormärz: 516 Paare, 302 Rels, 288 mit Rel (56%)
  Realismus und Gründerzeit: 535 Paare, 307 Rels, 294 mit Rel (55%)
  Die literarische Moderne (1890–1920): 688 Paare, 385 Rels, 368 mit Rel (53%)
  Literatur in der Weimarer Republik: 619 Paare, 383 Rels, 367 